# Evidential Deep Learning (EDL) Classifier Pipeline
This notebook trains a Dirichlet-based classifier under Evidential Deep Learning (EDL) and compares it with a deterministic Softmax baseline to inspect Expected Calibration Error (ECE).

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# Add parent folder to path to allow importing from src
sys.path.append(os.path.abspath(os.path.join('..')))

from src.dataset import SeaDronesSeeDataset
from src.models import EDLClassifier, SoftmaxClassifier, edl_loss, calculate_ece
from src.eda import generate_mock_coco_json

In [ ]:
local_paths = [
    "../data/annotations/instances_val.json",
    "../archive/compressed/annotations/instances_val.json",
    "../archive/annotations/instances_val.json",
    "../sds-dataset/annotations/instances_val.json"
]

DATASET_JSON = None
for path in local_paths:
    if os.path.exists(path):
        DATASET_JSON = os.path.abspath(path)
        break

if DATASET_JSON is None:
    DATASET_JSON = "../data/annotations/instances_val.json"
    generate_mock_coco_json(DATASET_JSON)

with open(DATASET_JSON, 'r') as f:
    coco = json.load(f)

categories = {cat["id"]: cat["name"] for cat in coco["categories"]}
img_dict = {img["id"]: img for img in coco["images"]}
cat_to_idx = {cid: idx for idx, cid in enumerate(sorted(categories.keys()))}

dataset = SeaDronesSeeDataset(coco["annotations"], img_dict, cat_to_idx, include_center_dist=False)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)
print(f"Loaded dataset with {len(dataset)} samples.")

In [ ]:
model = EDLClassifier(input_dim=6, num_classes=len(categories))
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Training Evidential Deep Learning Classifier...")
for epoch in range(1, 6):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        optimizer.zero_grad()
        evidence = model(x)
        alpha = evidence + 1.0
        y_onehot = torch.nn.functional.one_hot(y, num_classes=len(categories)).float()
        loss = edl_loss(alpha, y_onehot, epoch, len(categories))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(x)
        
    model.eval()
    with torch.no_grad():
        evidence = model(dataset.features)
        probs = (evidence + 1.0) / torch.sum(evidence + 1.0, dim=1, keepdim=True)
        val_ece = calculate_ece(probs.numpy(), dataset.labels.numpy())
        acc = np.mean(np.argmax(probs.numpy(), axis=1) == dataset.labels.numpy())
        
    print(f"Epoch {epoch}/5 | Loss: {total_loss/len(dataset):.4f} | Acc: {acc:.3f} | ECE: {val_ece:.4f}")

In [ ]:
# Train softmax baseline
softmax_model = SoftmaxClassifier(input_dim=6, num_classes=len(categories))
sm_optimizer = optim.Adam(softmax_model.parameters(), lr=0.01)
ce_loss_fn = torch.nn.CrossEntropyLoss()

print("Training Softmax Baseline...")
for epoch in range(1, 6):
    softmax_model.train()
    for x, y in train_loader:
        sm_optimizer.zero_grad()
        logits = softmax_model.net(x)
        ce_loss_fn(logits, y).backward()
        sm_optimizer.step()

softmax_model.eval()
with torch.no_grad():
    sm_probs = softmax_model(dataset.features).numpy()
    sm_ece   = calculate_ece(sm_probs, dataset.labels.numpy())
    sm_acc   = float((sm_probs.argmax(1) == dataset.labels.numpy()).mean())

# EDL calibration evaluation
model.eval()
with torch.no_grad():
    ev       = model(dataset.features)
    edl_probs = (ev + 1.0) / (ev + 1.0).sum(1, keepdim=True)
    edl_ece   = calculate_ece(edl_probs.numpy(), dataset.labels.numpy())
    edl_acc   = float((edl_probs.numpy().argmax(1) == dataset.labels.numpy()).mean())

q1_df = pd.DataFrame({
    "Model":     ["Softmax Baseline", "EDL (Ours)"],
    "Accuracy":  [f"{sm_acc:.3f}",   f"{edl_acc:.3f}"],
    "ECE (lower=better)":     [f"{sm_ece:.4f}",   f"{edl_ece:.4f}"],
    "Calibrated?": ["✗" if sm_ece > edl_ece else "✓",
                    "✓" if edl_ece < sm_ece  else "✗"],
})
print("\n=== Calibration Comparison ===")
print(q1_df.to_string(index=False))